In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_selection import (
    chi2, f_classif, mutual_info_classif, VarianceThreshold
)
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler, StandardScaler
import warnings
from kneed import KneeLocator
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import datetime

warnings.filterwarnings("ignore")

In [2]:
NUM_CORRIDA = 4
fecha_actual = datetime.datetime.today().strftime('%Y_%m_%d')
SUBFOLDER = f"{NUM_CORRIDA:02d}_{fecha_actual}"

REPO_ROOT = Path('.').resolve()
RUN_ID = datetime.datetime.today().strftime('%Y%m%d')

OUTPUT_PATH = REPO_ROOT / 'data' / 'intermediate' / '05_seleccion' / SUBFOLDER
METRIC_PATH = OUTPUT_PATH / 'metricas'
GRAPH_PATH = OUTPUT_PATH / 'graficos'
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
METRIC_PATH.mkdir(parents=True, exist_ok=True)
GRAPH_PATH.mkdir(parents=True, exist_ok=True)

INPUT_PATH = REPO_ROOT / 'data' / 'intermediate' / '04_featuring' / RUN_ID


In [3]:

def seleccionar_variables(X_train, y_train, metodo, auto_umbral=True, nombre_dataset=""):
    if metodo == "CHI2":
        X_train_nonneg = MinMaxScaler().fit_transform(X_train)
        scores = chi2(X_train_nonneg, y_train)[0]
    elif metodo == "ANOVA":
        scores = f_classif(X_train, y_train)[0]
    elif metodo == "MI":
        scores = mutual_info_classif(X_train, y_train)
    elif metodo == "VAR":
        threshold_var = 0.01  
        selector = VarianceThreshold(threshold=threshold_var)
        selector.fit(X_train)
        mask = selector.get_support()
        cols = X_train.columns[mask]
        scores_series = pd.Series(selector.variances_, index=X_train.columns)
        
        # Graficar varianzas
        with PdfPages(os.path.join(GRAPH_PATH, f"varianza_{nombre_dataset}_{metodo}.pdf")) as pdf:
            plt.figure(figsize=(10,4))
            scores_series.sort_values(ascending=False).plot(kind='bar')
            plt.title(f"Varianza de variables - {nombre_dataset} [{metodo}]")
            plt.ylabel("Varianza")
            plt.tight_layout()
            pdf.savefig(); plt.close()
        return cols, scores_series
    else:
        raise ValueError("Método de selección no reconocido")

    scores_series = pd.Series(scores, index=X_train.columns)
    scores_sorted = scores_series.sort_values(ascending=False)

    # Graficar curva de scores y guardar PDF
    with PdfPages(os.path.join(GRAPH_PATH, f"codo_{nombre_dataset}_{metodo}.pdf")) as pdf:
        plt.figure(figsize=(10,4))
        plt.plot(range(len(scores_sorted)), scores_sorted.values, marker='o')
        plt.xlabel("Variables ordenadas")
        plt.ylabel("Score")
        plt.title(f"Curva de importancia de variables - {nombre_dataset} [{metodo}]")
        plt.tight_layout()
        pdf.savefig(); plt.close()

    if auto_umbral:
        knee = KneeLocator(
            range(len(scores_sorted)), scores_sorted.values,
            curve='convex', direction='decreasing'
        )
        if knee.knee is not None and knee.knee > 0:
            best_n = knee.knee
            selected_features = scores_sorted.index[:best_n]
            # Guardar info del codo
            codo_info = {
                "metodo": metodo,
                "dataset": nombre_dataset,
                "codo_posicion": int(best_n),
                "score_codo": float(scores_sorted.values[best_n-1]),
                "total_variables": len(scores_sorted)
            }
        else:
            threshold = 0.05 * np.nanmax(scores)
            selected_mask = scores >= threshold
            selected_features = X_train.columns[selected_mask]
            codo_info = {
                "metodo": metodo,
                "dataset": nombre_dataset,
                "codo_posicion": None,
                "score_codo": None,
                "umbral_fallback": float(threshold),
                "total_variables": len(scores_sorted)
            }
    else:
        threshold = 0.05 * np.nanmax(scores)
        selected_mask = scores >= threshold
        selected_features = X_train.columns[selected_mask]
        codo_info = {
            "metodo": metodo,
            "dataset": nombre_dataset,
            "codo_posicion": None,
            "score_codo": None,
            "umbral_fallback": float(threshold),
            "total_variables": len(scores_sorted)
        }

    # Guardar info del codo en CSV
    pd.DataFrame([codo_info]).to_csv(
        os.path.join(METRIC_PATH, f"codo_{nombre_dataset}_{metodo}.csv"), index=False
    )

    return selected_features, scores_series


In [4]:

def aplicar_pca(X_train, X_test, n_components, modo="fijo", nombre_dataset=""):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    if modo == "fijo":
        pca_full = PCA().fit(X_train_scaled) 
        pca = PCA(n_components=n_components)
        X_train_pca = pca.fit_transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)
        scores = pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(n_components)])
        n_opt = n_components
    elif modo == "optimo":
        pca_full = PCA().fit(X_train_scaled)
        cumsum = np.cumsum(pca_full.explained_variance_ratio_)
        n_opt = np.argmax(cumsum >= 0.95) + 1
        pca = PCA(n_components=n_opt)
        X_train_pca = pca.fit_transform(X_train_scaled)
        X_test_pca = pca.transform(X_test_scaled)
        scores = pd.Series(pca.explained_variance_ratio_, index=[f"PC{i+1}" for i in range(n_opt)])
    else:
        raise ValueError("Modo PCA no reconocido")

    # Graficar varianza explicada acumulada
    with PdfPages(os.path.join(GRAPH_PATH, f"pca_{nombre_dataset}_{modo}.pdf")) as pdf:
        plt.figure(figsize=(10,4))
        plt.plot(np.cumsum(pca_full.explained_variance_ratio_), marker='o')
        plt.axhline(0.95, color='r', linestyle='--', label='95% varianza')
        plt.xlabel("N° componentes")
        plt.ylabel("Varianza explicada acumulada")
        plt.title(f"PCA - Varianza explicada acumulada [{nombre_dataset}]")
        plt.legend()
        plt.tight_layout()
        pdf.savefig(); plt.close()

    # Guardar info de PCA en CSV
    pca_info = {
        "dataset": nombre_dataset,
        "modo": modo,
        "componentes_optimos": n_opt,
        "varianza_95": float(np.cumsum(pca_full.explained_variance_ratio_)[n_opt-1])
    }
    pd.DataFrame([pca_info]).to_csv(
        os.path.join(METRIC_PATH, f"pca_{nombre_dataset}_{modo}.csv"), index=False
    )

    return pd.DataFrame(X_train_pca), pd.DataFrame(X_test_pca), scores


In [5]:

# Procesar todas las combinaciones
metodos = ["CHI2", "ANOVA", "MI", "VAR", "ALL"]
resumen_comparativo = []

procesados = set()

for carpeta in os.listdir(INPUT_PATH):
    carpeta_path = os.path.join(INPUT_PATH, carpeta)
    if not os.path.isdir(carpeta_path):
        continue

    for tipo in ["ORIGINAL", "FE"]:
        nombre_dataset = f"{carpeta}_{tipo}"
        if nombre_dataset in procesados:
            continue  # Evita duplicados
        procesados.add(nombre_dataset)
        try:
            X_train = pd.read_parquet(f"{carpeta_path}/X_train_{carpeta}_{tipo}.parquet")
            X_test = pd.read_parquet(f"{carpeta_path}/X_test_{carpeta}_{tipo}.parquet")
            y_train = pd.read_parquet(f"{carpeta_path}/y_train_{carpeta}_{tipo}.parquet")
            y_test = pd.read_parquet(f"{carpeta_path}/y_test_{carpeta}_{tipo}.parquet")
        except Exception as e:
            print(f"Error cargando {nombre_dataset}: {e}")
            continue

        print(f"Procesando {nombre_dataset}...")

        # Métodos clásicos de selección
        for metodo in metodos:
            if metodo == "CHI2" and (X_train < 0).any().any():
                print(f"Saltando CHI2 por contener negativos en {nombre_dataset}")
                continue

            if metodo == "ALL":
                selected_columns = X_train.columns
                scores = pd.Series([1.0] * len(selected_columns), index=selected_columns)
            elif metodo == "VAR":
                selected_columns, scores = seleccionar_variables(X_train, y_train.values.ravel(), metodo, auto_umbral=False, nombre_dataset=nombre_dataset)
            else:
                try:
                    selected_columns, scores = seleccionar_variables(X_train, y_train.values.ravel(), metodo, auto_umbral=True, nombre_dataset=nombre_dataset)
                except Exception as e:
                    print(f"Error aplicando {metodo} en {nombre_dataset}: {e}")
                    continue

            # Guardar métricas
            scores.to_csv(os.path.join(METRIC_PATH, f"metricas_{nombre_dataset}_{metodo}.csv"), header=["score"])

            # Guardar datasets
            X_train_sel = X_train[selected_columns]
            X_test_sel = X_test[selected_columns.intersection(X_test.columns)]
            X_train_sel.to_parquet(os.path.join(OUTPUT_PATH, f"X_train_{nombre_dataset}_{metodo}.parquet"))
            X_test_sel.to_parquet(os.path.join(OUTPUT_PATH, f"X_test_{nombre_dataset}_{metodo}.parquet"))
            y_train.to_parquet(os.path.join(OUTPUT_PATH, f"y_train_{nombre_dataset}_{metodo}.parquet"))
            y_test.to_parquet(os.path.join(OUTPUT_PATH, f"y_test_{nombre_dataset}_{metodo}.parquet"))

            print(f"✔ {nombre_dataset} - {metodo}: {len(selected_columns)} variables seleccionadas")

            resumen_comparativo.append({
                "dataset": nombre_dataset,
                "metodo": metodo,
                "columnas_seleccionadas": len(selected_columns)
            })

        # PCA30 y PCAopt
        for pca_tipo, modo in [("PCA30", "fijo"), ("PCAopt", "optimo")]:
            try:
                X_train_pca, X_test_pca, pca_scores = aplicar_pca(X_train, X_test, n_components=30, modo=modo, nombre_dataset=nombre_dataset)

                # Guardar datasets
                X_train_pca.to_parquet(os.path.join(OUTPUT_PATH, f"X_train_{nombre_dataset}_{pca_tipo}.parquet"))
                X_test_pca.to_parquet(os.path.join(OUTPUT_PATH, f"X_test_{nombre_dataset}_{pca_tipo}.parquet"))
                y_train.to_parquet(os.path.join(OUTPUT_PATH, f"y_train_{nombre_dataset}_{pca_tipo}.parquet"))
                y_test.to_parquet(os.path.join(OUTPUT_PATH, f"y_test_{nombre_dataset}_{pca_tipo}.parquet"))

                # Guardar métricas
                pca_scores.to_csv(os.path.join(METRIC_PATH, f"metricas_{nombre_dataset}_{pca_tipo}.csv"), header=["explained_variance_ratio"])

                print(f"✔ {nombre_dataset} - {pca_tipo}: {X_train_pca.shape[1]} componentes")
                resumen_comparativo.append({
                    "dataset": nombre_dataset,
                    "metodo": pca_tipo,
                    "columnas_seleccionadas": X_train_pca.shape[1]
                })
            except Exception as e:
                print(f"Error en PCA {pca_tipo} para {nombre_dataset}: {e}")


Procesando MaxAbs_ORIGINAL...
Saltando CHI2 por contener negativos en MaxAbs_ORIGINAL
✔ MaxAbs_ORIGINAL - ANOVA: 41 variables seleccionadas
✔ MaxAbs_ORIGINAL - MI: 29 variables seleccionadas
✔ MaxAbs_ORIGINAL - VAR: 139 variables seleccionadas
✔ MaxAbs_ORIGINAL - ALL: 545 variables seleccionadas
✔ MaxAbs_ORIGINAL - PCA30: 30 componentes
✔ MaxAbs_ORIGINAL - PCAopt: 475 componentes
Procesando MaxAbs_FE...
Saltando CHI2 por contener negativos en MaxAbs_FE
✔ MaxAbs_FE - ANOVA: 10 variables seleccionadas
✔ MaxAbs_FE - MI: 17 variables seleccionadas
✔ MaxAbs_FE - VAR: 354 variables seleccionadas
✔ MaxAbs_FE - ALL: 1162 variables seleccionadas
✔ MaxAbs_FE - PCA30: 30 componentes
✔ MaxAbs_FE - PCAopt: 481 componentes
Procesando MinMax_ORIGINAL...
✔ MinMax_ORIGINAL - CHI2: 19 variables seleccionadas
✔ MinMax_ORIGINAL - ANOVA: 41 variables seleccionadas
✔ MinMax_ORIGINAL - MI: 29 variables seleccionadas
✔ MinMax_ORIGINAL - VAR: 139 variables seleccionadas
✔ MinMax_ORIGINAL - ALL: 545 variables s

In [6]:

# Guardar resumen general
df_resumen = pd.DataFrame(resumen_comparativo)
df_resumen.to_csv(os.path.join(OUTPUT_PATH, "resumen_cantidad_variables.csv"), index=False)